# Pré-processamento dos dados

---

## 1. Carregando e separando os dados

In [1]:
RANDOM_SEED = 42

In [2]:
import pandas as pd

leads = pd.read_csv('data/churn_aplicacao_leads_final.csv')
pedidos = pd.read_csv('data/churn_comportamental_pedidos_final.csv')

total = leads.merge(right=pedidos, on='id_cliente', how='left')

Vamos treinar dois modelos: um **comportamental** e um de **análise de leads**.

No modelo comportamental, o conjunto de treinamento será referente aos clientes que tem pedidos correspondentes na tabela de pedidos, enquanto que o conjunto de treinamento do modelo de análise de leads conterá apenas dados de clientes que nunca fizeram um pedido antes. Nesse último caso, nenhum atributo da tabela de pedidos será relevante, então podemos também removê-los dos atributos disponíveis.

Com base nisso, separamos os conjuntos usando essas regras:

In [4]:
leads_raw: pd.DataFrame = total[total.id_pedido.isna()]           # Clientes que nunca pediram
comportamental_raw: pd.DataFrame = total[total.id_pedido.notna()] # Clientes que tem pedidos

leads_raw = leads_raw.drop(['id_pedido', 'valor', 'produtos', 'data_entrega'], axis='columns') # Removendo atributos referentes a pedidos

Por fim, vamos definir uma porcentagem para a divisão entre treino + validação e teste:

In [5]:
TRAIN_VAL_RATIO = 0.8
TEST_RATIO      = 0.2

### 1.1. Separando o conjunto de dados `leads_raw`

Sem levar os pedidos em consideração, temos um conjunto de dados que não apresenta uma relação de temporalidade relevante, então a sua divisão segue de modo padrão: faremos uma divisão estratificada no atributo-alvo, o `churned`:

In [6]:
from sklearn.model_selection import train_test_split

leads_raw_X, leads_raw_y = leads_raw, leads_raw.pop('churned')

leads_raw_train_val_X, leads_raw_test_X, leads_raw_train_val_y, leads_raw_test_y = train_test_split(
    leads_raw_X,
    leads_raw_y,
    train_size=TRAIN_VAL_RATIO,
    test_size=TEST_RATIO,
    random_state=RANDOM_SEED,
    stratify=leads_raw_y
)

LEADS = pd.DataFrame({ # pondo tudo num DataFrame só pela facilidade de acesso
    'train_val': {
        'X': leads_raw_train_val_X,
        'y': leads_raw_train_val_y,
    },
    'test': {
        'X': leads_raw_test_X,
        'y': leads_raw_test_y,
    },
})

print(f'Tamanho do conjunto de treino + validação: {LEADS.train_val.X.shape[0]} linhas x {LEADS.train_val.X.shape[1]} colunas')
print(f'Tamanho do conjunto de teste:              {LEADS.test.X.shape[0]} linhas x {LEADS.test.X.shape[1]} colunas')

Tamanho do conjunto de treino + validação: 800 linhas x 18 colunas
Tamanho do conjunto de teste:              200 linhas x 18 colunas


### 1.2. Separando o conjunto de dados `comportamental_raw`

Como o conjunto de dados comportamental leva em consideração os pedidos dos clientes, temos uma série temporal, como discutido no processo de EDA.

Para separar os dados agora, é necessário evitar que o futuro seja usado como treinamento para testes no passado, então, devemos ordenar os dados com base na data. A partir disso, basta escolher uma data de corte e separar os dois conjuntos adequadamente.

Queremos manter a mesma proporção de amostras de teste que o conjunto de análise de leads, então usaremos `TEST_RATIO` do intervalo total (90 dias) para definir a janela de corte:

In [7]:
TEST_TIMEFRAME = pd.to_timedelta(90 * TEST_RATIO, 'days')

Agora, basta separar ambos conjuntos:

In [8]:
comportamental_raw.data_entrega = comportamental_raw.data_entrega.apply(pd.to_datetime)

ordered = comportamental_raw.sort_values(by='data_entrega')

date_threshold = pd.to_datetime(ordered.data_entrega.max()) - TEST_TIMEFRAME

comportamental_raw_train_val = ordered[ordered.data_entrega < date_threshold]
comportamental_raw_test = ordered[ordered.data_entrega >= date_threshold]

COMPORTAMENTAL = pd.DataFrame({ # pondo tudo num DataFrame só pela facilidade de acesso
    'train_val': {
        'X': comportamental_raw_train_val,
        'y': comportamental_raw_train_val.pop('churned'),
    },
    'test': {
        'X': comportamental_raw_test,
        'y': comportamental_raw_test.pop('churned'),
    },
})

print(f'Tamanho do conjunto de treino + validação: {COMPORTAMENTAL.train_val.X.shape[0]} linhas x {COMPORTAMENTAL.train_val.X.shape[1]} colunas')
print(f'Tamanho do conjunto de teste:              {COMPORTAMENTAL.test.X.shape[0]}  linhas x {COMPORTAMENTAL.test.X.shape[1]} colunas')

Tamanho do conjunto de treino + validação: 3554 linhas x 22 colunas
Tamanho do conjunto de teste:              887  linhas x 22 colunas


Verificando as proporções finais e os intervalos de tempo a serem utilizados:

In [9]:
print('Porcentagem da divisão dos dados comportamentais:')

print(f'{100 * COMPORTAMENTAL.train_val.X.shape[0] / comportamental_raw.shape[0]:.2f}% para treino e validação')
print(f'       |-> Intervalo de tempo: {COMPORTAMENTAL.train_val.X.data_entrega.nunique()} dias')

print(f'{100 * COMPORTAMENTAL.test.X.shape[0] / comportamental_raw.shape[0]:.2f}% para teste')
print(f'       |-> Intervalo de tempo: {TEST_TIMEFRAME.days} dias')

Porcentagem da divisão dos dados comportamentais:
80.03% para treino e validação
       |-> Intervalo de tempo: 73 dias
19.97% para teste
       |-> Intervalo de tempo: 18 dias


---

# 2. Pré-processamento dos dados

### 2.1. Conjunto de análise de leads

Aqui vamos definir as etapas de pré-processamento para todos os dados referentes ao cliente isoladamente, sem levar em conta seu comportamento. 

O pipeline que criarmos no final será reutilizado para o conjunto comportamental!

#### 2.1.1. Removendo atributos

Começamos removendo os seguintes atributos:
* `id_cliente`: identificador irrelevante
* `uf`: não existe mais de uma UF no conjunto de dados
* `google_maps_avaliacao`, `google_maps_contagem_avaliacoes`: faltante em quase 99% das amostras, não têm semântica clara para nulo
* `cnae_codigo`: redundante com `segmento_cliente`. Na EDA, vimos que a maioria dos segmentos se concentra em uma única divisão do CNAE, mas o contrário não acontece (a divisão 53, por exemplo, reúne vários segmentos com taxas de churn entre 33% e 55%). Ou seja, o segmento já carrega a informação do CNAE e ainda separa melhor o churn

*Obs.: Os atributos `ifood_contagem_avaliacoes` e `ifood_faixa_preco` serão removidos no próximo passo.*

In [10]:
TO_DROP = ['id_cliente', 'uf', 'google_maps_avaliacao', 'google_maps_contagem_avaliacoes','cnae_codigo']

#### 2.1.2. Extrair features dos valores do iFood/Google Maps

Vamos remover os atributos que carregam informações mais específicas do cliente acerca da sua presença no iFood e/ou Google Maps, mas ainda queremos saber se o cliente está no iFood ou não, e se tem website no Google Maps ou não.

Para isso, vamos corrigir os valores nulos de `google_maps_tem_website` e criar uma nova feature, `tem_ifood`:

In [11]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer


def extract_tem_ifood(X: pd.DataFrame):
    X['tem_ifood'] = (X.ifood_faixa_preco.notna() | X.ifood_contagem_avaliacoes.notna()).astype(int)
    X.drop(columns=['ifood_contagem_avaliacoes', 'ifood_faixa_preco'], inplace=True)
    
    return X

IFOOD_FEATURE_EXTRACTOR = FunctionTransformer(extract_tem_ifood, feature_names_out=lambda _, __: ['tem_ifood'])

# sabemos que google_maps_tem_website ou é 1 ou é nan,
# então só basta imputar onde tem nan => 0
GOOGLE_MAPS_IMPUTER = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=0)

#### 2.1.3. Lidando com faixas de valores

Já que vamos remover os atributos `ifood_contagem_avaliacoes`, `google_maps_avaliacao` e `google_maps_contagem_avaliacoes`, os atributos que ainda representam faixas de valores no conjunto de leads são **`capital_social`** e **`idade_cnpj`**.

Nesses casos, vamos fazer um *label encoding* desses valores para manter a relação de ordem entre intervalos:

In [12]:
from sklearn.preprocessing import OrdinalEncoder


def get_lower_bound(range: str) -> float:
    lower_bound = range.strip('()[]').split(',')[0]
    return float(lower_bound)

def get_ordered_ranges(data: pd.Series):
    return sorted(data.dropna().unique(), key=get_lower_bound)

CAPITAIS_SOCIAIS = get_ordered_ranges(LEADS.train_val.X.capital_social)
IDADES_CNPJ = get_ordered_ranges(LEADS.train_val.X.idade_cnpj)

CAPITAL_SOCIAL_ENCODER = OrdinalEncoder(categories=[CAPITAIS_SOCIAIS])
IDADE_CNPJ_ENCODER = OrdinalEncoder(categories=[IDADES_CNPJ])

#### 2.1.4. Lidando com valores numéricos com outliers estatísticos

Como visto na EDA, os atributos `serasa_contagem_negativacoes` e `serasa_contagem_protestos` têm valores geralmente perto de 0, mas possuem alguns outliers estatísticos muito acima desse nível.

Não queremos remover esses valores, então, como aprendemos nas aulas de EDA, aplicaremos uma função logaritmo nesses valores:

In [13]:
# vamos usar a função do numpy correspondente a f(x) = ln(1+x)
# pois os valores de x podem ser 0 e queremos valores reais
LOG_TRANSFORMER = FunctionTransformer(np.log1p, feature_names_out='one-to-one')

#### 2.1.5. Agrupando categorias de valores

O atributo `natureza_juridica` tem duas categorias primárias e outras duas que tem uma característica comum com as majoritárias. Podemos agrupar as categorias `230-5` e `213-5`, e as categorias `206-2` e `224-0`, conforme analisamos na EDA:

In [14]:
def transform_natureza_juridica(X: pd.DataFrame):
    X.natureza_juridica = X.natureza_juridica.str[:5].isin(['206-2', '224-0']).astype(int)
    return X

NATUREZA_JURIDICA_TRANSFORMER = FunctionTransformer(transform_natureza_juridica, feature_names_out='one-to-one')

#### 2.1.7. Extraindo informações de `serasa_credores`

Vamos extrair a quantidade de credores na "lista" da coluna:

In [15]:
import re


def extract_qtd_credores(X: pd.DataFrame):
    X.serasa_credores = X.serasa_credores.apply(lambda str: len(re.split(r',\s', str)) if pd.notna(str) else 0)
    return X

QTD_CREDORES_EXTRACTOR = FunctionTransformer(extract_qtd_credores, feature_names_out='one-to-one')

# TODO: definir se vamos fazer um encoding do tipo de credor que o cliente possui
# menos de 10% dos clientes no geral tinham credor.. vale a pena?

# na minha opinião não, além de que nenhuma váriavel se relacionou com churn no EDA, acho que deixaria grupos muitos pequenos

# TODO: decidir se mantemos serasa_socio_tem_negativacao --> vamos manter

#### 2.1.8. Codificando outras variáveis categóricas

- `fonte_cliente`: faremos um one-hot encoding simples, são apenas 5 categorias para essa variável;

In [16]:
from sklearn.preprocessing import OneHotEncoder

FONTES_CLIENTE = [f'Fonte {n}' for n in range(1, 6)]
FONTE_CLIENTE_ENCODER = OneHotEncoder(categories=[FONTES_CLIENTE], feature_name_combiner=lambda feature, val: f'{feature}_{val[-1]}')

- `segmento_cliente`: faremos um one-hot encoding apenas com os 14 segmentos que tinham pelo menos 30 leads na EDA. Os 7 segmentos mais raros irá funcionar como uma categoria "Outros", já que estimar cada um separadamente, com tão poucos leads, faria o modelo aprender ruído;

In [17]:
# segmentos com pelo menos 30 leads,com base na EDA -->14 dos 21
SEGMENTOS = [f'Segmento {n}' for n in [2, 3, 4, 5, 8, 10, 11, 13, 14, 15, 16, 18, 19, 20]]

SEGMENTO_ENCODER = OneHotEncoder(categories=[SEGMENTOS],handle_unknown='ignore',feature_name_combiner=lambda feature, val: f'{feature}_{val.split()[-1]}')

- `municipio`: agrupamos os 31 municípios em 3 regiões (Fortaleza,Região Metropolitana de Fortaleza (RMF) e Interior), seguindo a composição oficial da RMF, e depois aplicamos um one-hot encoding com essas 3 categorias.

In [18]:
from sklearn.pipeline import Pipeline

RMF= ['CAUCAIA', 'MARACANAU', 'MARANGUAPE', 'PACATUBA', 'ITAITINGA', 'EUSEBIO', 'AQUIRAZ', 'GUAIUBA', 'HORIZONTE','PACAJUS', 'CHOROZINHO', 'SAO GONCALO DO AMARANTE', 'PINDORETAMA', 'CASCAVEL', 'PARACURU', 'PARAIPABA','TRAIRI', 'SAO LUIS DO CURU']

def transform_municipio(X: pd.DataFrame):
    X['municipio'] = np.select(
        [X['municipio'] == 'FORTALEZA', X['municipio'].isin(RMF)],
        ['Fortaleza', 'RMF'],
        'Interior',
    )
    return X

# pipeline já que são dois passos em sequência
REGIAO_PIPELINE = Pipeline([
    ('regiao', FunctionTransformer(transform_municipio, feature_names_out='one-to-one')),
    ('onehot', OneHotEncoder(categories=[['Fortaleza', 'RMF', 'Interior']],feature_name_combiner=lambda feature, val: f'regiao_{val}')),
])

#### 2.1.9. Transformação dos dados

Vale salientar que essas transformações **não incluem** feature scaling ainda.

In [19]:
from sklearn.compose import ColumnTransformer

LEADS_PREPROCESSOR = ColumnTransformer(
    transformers=[
        # nome da etapa                    transformador usado            # colunas afetadas
        ('drop',                           'drop',                        TO_DROP),
        ('extract_tem_ifood',              IFOOD_FEATURE_EXTRACTOR,       ['ifood_contagem_avaliacoes', 'ifood_faixa_preco']),
        ('impute_google_maps_tem_website', GOOGLE_MAPS_IMPUTER,           ['google_maps_tem_website']),
        ('encode_capital_social',          CAPITAL_SOCIAL_ENCODER,        ['capital_social']),
        ('encode_idade_cnpj',              IDADE_CNPJ_ENCODER,            ['idade_cnpj']),
        ('transform_contagens_serasa',     LOG_TRANSFORMER,               ['serasa_contagem_negativacoes', 'serasa_contagem_protestos']),
        ('transform_natureza_juridica',    NATUREZA_JURIDICA_TRANSFORMER, ['natureza_juridica']),
        ('extract_qtd_credores',           QTD_CREDORES_EXTRACTOR,        ['serasa_credores']),
        ('encode_segmento_cliente',        SEGMENTO_ENCODER,              ['segmento_cliente']),
        ('encode_fonte_cliente',           FONTE_CLIENTE_ENCODER,         ['fonte_cliente']),
        ('encode_regiao',                  REGIAO_PIPELINE,               ['municipio']),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

LEADS_PREPROCESSOR

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('drop', ...), ('extract_tem_ifood', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for fe

### 2.2. Conjunto comportamental 

Agora vamos nos interessar com as informações que podemos extrair dos dados comportamentais.

#### 2.2.1. Garantindo a ordem cronológica

A grande ideia do pré-processamento desse conjunto é levar em consideração a temporalidade dos eventos. Nossa ideia é a seguinte:
- Consideraremos uma janela de tempo de `d` dias;
- Agruparemos o nosso conjunto de dados em janelas de `d` dias;
- Para cada cliente, calcularemos as métricas que queremos extrair dos dados comportamentais nos últimos `d` dias;
- Juntaremos os dados resultantes e aplicaremos as transformações que definimos para a análise de leads.

Para isso, é preciso garantir que os dados estarão organizados no tempo, para que uma janela de tempo não possua dados de uma janela futura. Então, criamos uma função auxiliar para o pipeline de transformação dos dados:

In [20]:
def sort_by_date(X: pd.DataFrame):
    X.data_entrega = X.data_entrega.apply(pd.to_datetime)
    X = X.sort_values(by='data_entrega').reset_index(drop=True)

    return X

COMPORTAMENTAL_SORTER = FunctionTransformer(sort_by_date, feature_names_out='one-to-one')

#### 2.2.1. Definindo as informações a extrair

Aqui definimos as métricas que vamos extrair dos dados comportamentais.

Para início de conversa, vamos definir os atributos que irão ser removidos, pois não serão úteis após os agrupamentos:

In [21]:
TO_DROP = ['id_pedido', 'data_entrega']

- Atributos em formato de intervalo (`valor` e `produtos`): queremos extrair o valor mais comum, a moda, de todos os pedidos feitos por um cliente em uma determinada janela de tempo.
    - No caso em que os dados são multimodais, por exemplo, um cliente fez 5 pedidos de valor `(0, 500]`, escolhemos o intervalo com o menor valor máximo, pois é a tendência do conjunto de dados;
    - Sabemos da EDA que existem alguns valores faltantes. Imputaremos o menor intervalo para esses valores pela mesma razão descrita acima.

In [96]:
VALORES = get_ordered_ranges(COMPORTAMENTAL.train_val.X.valor)
PRODUTOS = get_ordered_ranges(COMPORTAMENTAL.train_val.X.produtos)

def get_mode_pipeline(values: list[str]) -> Pipeline:
    return Pipeline(steps=[
        # troque valores faltantes pelo valor mais frequente...
        ("impute_mode",    SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
        # se não houver valores, use o menor valor dos intervalos...
        ("impute_no_mode", SimpleImputer(missing_values=0, strategy="constant", fill_value=values[0], keep_empty_features=True)),
        # encoding preservando a ordem
        ("encode_values",  OrdinalEncoder(categories=[values])),
    ])

def extract_mode(X: pd.Series) -> int:
    modes: list[int] = sorted(X.mode())

    # Como é mais comum que os pedidos tenham valor/produtos menores,
    # pego o menor valor se o conjunto for multimodal
    # Se o cliente não tem valores, retornar o menor possível
    return min(modes, default=0)

- Frequência de pedidos: vimos na EDA que existe uma correlação significante entre a frequência de pedidos de um cliente (ou seja, o intervalo médio entre entregas de pedidos de um cliente) e a taxa de churn.
- Analogamente, observamos que a variância da frequência dos pedidos tem um impacto na taxa de churn, então usaremos esse atributo também.

A partir disso, criamos funções para extrair essas informações de uma série dos atributos adequados:

In [35]:
def extract_freq_pedidos(data_entrega: pd.Series) -> float:
    days = data_entrega.drop_duplicates().sort_values()

    if len(days) == 1:
        # se só tiver um pedido, a frequência é 0
        return 0.0

    return days.diff().dt.days.mean()

In [36]:
def extract_variancia_freq(data_entrega: pd.Series, freq_pedidos: pd.Series) -> float:
    days = data_entrega.drop_duplicates().sort_values()

    if len(days) == 1:
        # se só tiver um pedido, não há intervalos para variar
        return 0.0

    intervalos = days.diff().dt.days.dropna()

    # variância populacional: média do quadrado do desvio de cada intervalo
    # em relação à frequência média (que já foi calculada)
    return ((intervalos - freq_pedidos.iloc[0]) ** 2).mean() #.mean() remove o NaN

Juntamos essas etapas em uma única função a ser chamada mais na frente pela pipeline de transformação dos dados:

In [73]:
NEW_FEATURES = ['num_pedidos', 'freq_pedidos', 'variancia_freq']

def extract_features(X: pd.DataFrame):
    X.valor = get_mode_pipeline(VALORES).fit_transform(pd.DataFrame(X.valor))
    X.produtos = get_mode_pipeline(PRODUTOS).fit_transform(pd.DataFrame(X.produtos))

    X.valor = extract_mode(X.valor)
    X.produtos = extract_mode(X.produtos)

    X['num_pedidos'] = X.data_entrega.count()
    X['freq_pedidos'] = extract_freq_pedidos(X.data_entrega)
    X['variancia_freq'] = extract_variancia_freq(X.data_entrega, X.freq_pedidos) # TODO

    X.drop(TO_DROP, axis='columns', inplace=True)
    X.drop_duplicates(inplace=True)
    
    return X

Como a nossa ideia é agregar esses informações para cada cliente, a função que realmente extrai esses dados primeiramente os agrupa pelo ID dos cliente e, em seguida, chama a função auxiliar que criamos acima:

In [51]:
def extract_by_id_cliente(X: pd.DataFrame):
    return X.groupby('id_cliente', as_index=False)[X.columns]  \
            .apply(extract_features)                         \
            .sort_index(ignore_index=True)

Por fim, precisamos criar uma função que, dado um tamanho de janela de tempo arbitrário (de `d` dias), consiga separar o conjunto de dados em grupos de `d` dias e extrair essas informações agregadas por cliente em cada grupo. 

Admitimos que, se `d` não divide o número total `N` de dias do conjunto, iremos descartar `N % d` dias do conjunto, para garantir que não existam janelas de tempo de tamanho desproporcional.

In [83]:
def aggregate_data(X: pd.DataFrame, timeframe_in_days: int):
    timedelta = pd.to_timedelta(timeframe_in_days, 'days')
    total_days: int = X.data_entrega.nunique()

    remainder = pd.to_timedelta(total_days % timeframe_in_days, 'days')
    print(f'Dividindo {total_days} dias em {total_days // timeframe_in_days} janelas de {timeframe_in_days} dias.')

    # desconsidere os primeiros dias se eles não caberem em uma janela
    current_day: pd.Timestamp = X.data_entrega.min() + remainder
    print(f'Sobrarão {remainder.days} dias, começando em {current_day.date()}...')

    groups = []
    while current_day < X.data_entrega.max():
        next_day = current_day + timedelta

        # a janela é o intervalo [CURRENT_DAY, NEXT_DAY)
        mask = (X.data_entrega >= current_day) & (X.data_entrega < next_day)
        groups.append(extract_by_id_cliente(X[mask]))

        print(f'Janela {len(groups)}: [{current_day.date()} -> {next_day.date()}) ({groups[-1].shape[0]} amostras)')
        current_day = next_day
        
    return pd.concat(groups, ignore_index=True)

Criamos uma função que retorna o transformer final com base no "`d`" que escolhermos:

In [41]:
def get_feature_names(_, original_features: np.ndarray):
    original_features: list[str] = original_features.tolist()

    for dropped in TO_DROP:
        original_features.remove(dropped)

    return original_features + NEW_FEATURES

def ComportamentalFeatureTransformer(timeframe_in_days: int) -> FunctionTransformer:
    return FunctionTransformer(
        aggregate_data,
        kw_args={ 'timeframe_in_days': timeframe_in_days },
        feature_names_out=get_feature_names
    )

#### 2.2.2. Transformação dos dados

Por fim, a transformação dos dados final é função do tamanho da janela de tempo que escolhermos, e ela concatenará as etapas descritas anteriormente.

Como dito, reutilizaremos as transformações da etapa de leads:

In [42]:
def get_comportamental_preprocessor(timeframe_in_days: int) -> Pipeline:
    COMPORTAMENTAL_FEATURE_EXTRACTOR = ComportamentalFeatureTransformer(timeframe_in_days)
    
    return Pipeline(steps=[
            ('sort',                        COMPORTAMENTAL_SORTER),
            ('extract_features', COMPORTAMENTAL_FEATURE_EXTRACTOR),
            ('preprocess_leads',               LEADS_PREPROCESSOR),
        ],
    )

# Exemplo: caso fossemos testar com uma janela de 14 dias...
get_comportamental_preprocessor(14)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('sort', ...), ('extract_features', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function sor...001B2B4BD4670>
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",'one-to-one'
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"kw_args kw_args: dict, default